# Qwen3.5-27B Post-LoRA L50 Causal-Subspace Decomposition

[实验设置与运行说明](../../../docs/experiment_notes/07_appendix_lora_training/e4352c84c2536886.md)


In [ ]:
# Run once. Every later code cell reports progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 16 tracked cells.')


## 1. Freeze a genuinely new test

The previous torque-control result motivates the hypothesis but its scenes are never reused for the new confirmatory decision.


In [ ]:
%%cell_progress 1/16 Mount Drive and select a complete VisualTorque project
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Auditing candidate project roots')
from pathlib import Path
import shutil

PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
REQUIRED_TOOLKIT_PATHS = (
    'pyproject.toml',
    'src/visual_torque_toolkit/__init__.py',
    'src/visual_torque_toolkit/contracts.py',
    'src/visual_torque_toolkit/classify.py',
    'src/visual_torque_toolkit/dataset.py',
    'src/visual_torque_toolkit/edits.py',
    'src/visual_torque_toolkit/generate.py',
    'src/visual_torque_toolkit/prompts.py',
    'src/visual_torque_toolkit/render.py',
    'src/visual_torque_toolkit/splits.py',
    'src/visual_torque_toolkit/world.py',
    'src/visual_torque_toolkit/behavior/__init__.py',
    'src/visual_torque_toolkit/behavior/adapter.py',
    'src/visual_torque_toolkit/behavior/data.py',
    'src/visual_torque_toolkit/behavior/schema.py',
)

def missing_toolkit_paths(project_root):
    toolkit_root = project_root / 'toolkit'
    return [
        relative for relative in REQUIRED_TOOLKIT_PATHS
        if not (toolkit_root / relative).is_file()
    ]

candidate_audit = {
    str(candidate): missing_toolkit_paths(candidate)
    for candidate in PROJECT_CANDIDATES
}
complete_candidates = [
    candidate for candidate in PROJECT_CANDIDATES
    if not candidate_audit[str(candidate)]
]
assert complete_candidates, (
    'No complete VisualTorque toolkit was found. '
    f'Missing-file audit: {candidate_audit}'
)
PROJECT_ROOT = complete_candidates[0]

cell_stage('Copying immutable toolkit source to local SSD')
DRIVE_TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
TOOLKIT_ROOT = Path('/content/visualtorque_toolkit_runtime')
if TOOLKIT_ROOT.exists():
    shutil.rmtree(TOOLKIT_ROOT)
shutil.copytree(DRIVE_TOOLKIT_ROOT, TOOLKIT_ROOT)
print('Project:', PROJECT_ROOT)
print('Toolkit runtime:', TOOLKIT_ROOT)


In [ ]:
%%cell_progress 2/16 Install and verify the frozen causal environment
import importlib
import importlib.metadata
import json
import os
import subprocess
import sys

cell_stage('Configuring caches before importing Torch')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('HF_HOME', '/content/hf_cache')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

cell_stage('Removing incompatible optional Torch packages')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio', 'torchao'],
    check=False,
)
importlib.invalidate_caches()

cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    '--upgrade-strategy', 'only-if-needed',
    'transformers==5.14.0', 'peft==0.20.0', 'accelerate>=1.10',
    'sentencepiece', 'protobuf', 'huggingface_hub>=0.34',
    'safetensors', 'tqdm>=4.66', 'scikit-learn>=1.5',
    'scipy>=1.13', 'pandas>=2.2', 'matplotlib>=3.8',
    'joblib>=1.4', 'pillow>=10.4',
])

cell_stage('Importing the immutable toolkit copy')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if module_name == 'visual_torque_toolkit' or module_name.startswith('visual_torque_toolkit.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import visual_torque_toolkit
from IPython.display import display
from peft import PeftModel
from scipy.stats import binomtest
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter
from visual_torque_toolkit.generate import (
    GenerationConfig, generate_multi_stack_scenes,
)
from visual_torque_toolkit.prompts import QuestionKind, generate_questions
from visual_torque_toolkit.render import RenderStyle, render_scene

# Defensive Qwen3.5 compatibility patch for exact candidate scoring.
def append_text_prefix_aligned(self, inputs, prefix):
    if not prefix:
        return inputs
    batch_size = inputs['input_ids'].shape[0]
    prefix_tensor = torch.tensor(
        prefix,
        dtype=inputs['input_ids'].dtype,
        device=inputs['input_ids'].device,
    ).unsqueeze(0).expand(batch_size, -1)
    inputs['input_ids'] = torch.cat((inputs['input_ids'], prefix_tensor), dim=1)
    attention_extension = torch.ones(
        (batch_size, len(prefix)),
        dtype=inputs['attention_mask'].dtype,
        device=inputs['attention_mask'].device,
    )
    inputs['attention_mask'] = torch.cat(
        (inputs['attention_mask'], attention_extension), dim=1
    )
    for key in ('token_type_ids', 'mm_token_type_ids'):
        if key not in inputs:
            continue
        extension = torch.zeros(
            (batch_size, len(prefix)),
            dtype=inputs[key].dtype,
            device=inputs[key].device,
        )
        inputs[key] = torch.cat((inputs[key], extension), dim=1)
    inputs.pop('position_ids', None)
    return inputs

HuggingFaceMultimodalAdapter._append_text_prefix = append_text_prefix_aligned
torch.backends.cuda.matmul.allow_tf32 = True
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents
try:
    torchao_version = importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError:
    torchao_version = None
assert torchao_version is None
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('transformers:', importlib.metadata.version('transformers'))
print('peft:', importlib.metadata.version('peft'))
print('Toolkit:', toolkit_path)


In [ ]:
%%cell_progress 3/16 Verify prior evidence and configure the frozen follow-up
cell_stage('Checking GPU and immutable prerequisites')
import hashlib
from collections import Counter, defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone
from zipfile import ZipFile
from scipy.stats import binomtest

assert torch.cuda.is_available(), 'Select a GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
assert GPU_MEMORY_GIB >= 88, (GPU_NAME, GPU_MEMORY_GIB)

MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
EXPECTED_LAYER_COUNT = 64
L50 = 50
SEED = 20260907
CANDIDATE_COUNT = 20000
SCENES_PER_CONFLICT_CELL = 24
DEV_SCENES_PER_CELL = 12
TEST_SCENES_PER_CELL = 12
PATCH_BATCH_SIZE = 4
BOOTSTRAP_REPLICATES = 10000
SIGN_FLIP_REPLICATES = 10000
RANDOM_SUBSPACE_COUNT = 32
NONINFERIORITY_FRACTION = 0.80
ALPHA_LEVEL = 0.05
RUN_VERSION = 'v1'

JOINT_RANKS = (1, 2, 4, 8, 16, 32, 48, 64)
SHARED_RANKS = (1, 2, 4, 8, 16, 24)
MAPPING_RANKS = (1, 2, 4, 8, 16, 32, 48)

TRAIN_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_bf16_lora_weight_shortcut_v1'
ADAPTER_DIR = TRAIN_ROOT / 'adapter_final'
ADAPTER_MANIFEST_PATH = TRAIN_ROOT / 'ADAPTER_MANIFEST.json'
DISCOVERY_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_base_vs_lora_mechanism_discovery_v1'
DISCOVERY_COMPLETE_PATH = DISCOVERY_ROOT / 'RUN_COMPLETE.json'
FROZEN_HYPOTHESES_PATH = DISCOVERY_ROOT / 'FROZEN_MECHANISM_HYPOTHESES.json'
LORA_CAPTURE_DIR = DISCOVERY_ROOT / 'lora_capture'
BLIND_V1_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_postlora_blind_causal_validation_v1'
BLIND_V1_DATASET_DIR = BLIND_V1_ROOT / 'blind_dataset'
STRESS_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_postlora_torque_control_stresstest_v1'
STRESS_COMPLETE_PATH = STRESS_ROOT / 'RUN_COMPLETE.json'
STRESS_DECISION_PATH = STRESS_ROOT / 'analysis' / 'torque_control_decision.json'

for required in (
    ADAPTER_DIR / 'adapter_config.json', ADAPTER_MANIFEST_PATH,
    DISCOVERY_COMPLETE_PATH, FROZEN_HYPOTHESES_PATH,
    LORA_CAPTURE_DIR / 'capture_config.json',
    BLIND_V1_ROOT / 'RUN_COMPLETE.json',
    BLIND_V1_DATASET_DIR / 'scenes.jsonl',
    STRESS_COMPLETE_PATH, STRESS_DECISION_PATH,
):
    assert required.is_file(), required

def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if torch.is_tensor(value) and value.numel() == 1:
        return value.item()
    raise TypeError(f'Cannot serialize {type(value).__name__}')

def write_json_atomic(file_path, value):
    file_path = Path(file_path)
    temporary = file_path.with_suffix(file_path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(
            value, indent=2, ensure_ascii=False, sort_keys=True,
            default=json_default,
        ) + '\n',
        encoding='utf-8',
    )
    temporary.replace(file_path)

def write_jsonl_atomic(file_path, rows):
    file_path = Path(file_path)
    temporary = file_path.with_suffix(file_path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(
                row, ensure_ascii=False, sort_keys=True,
                default=json_default,
            ) + '\n')
    temporary.replace(file_path)

def read_jsonl(file_path):
    with Path(file_path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def sha256_file(file_path):
    digest = hashlib.sha256()
    with Path(file_path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value!r}'.encode('utf-8')).hexdigest()

adapter_manifest = json.loads(ADAPTER_MANIFEST_PATH.read_text(encoding='utf-8'))
frozen_hypotheses = json.loads(FROZEN_HYPOTHESES_PATH.read_text(encoding='utf-8'))
stress_decision = json.loads(STRESS_DECISION_PATH.read_text(encoding='utf-8'))
assert adapter_manifest['base_model_id'] == MODEL_ID
assert adapter_manifest['base_model_revision'] == MODEL_REVISION
assert frozen_hypotheses['model_id'] == MODEL_ID
assert frozen_hypotheses['model_revision'] == MODEL_REVISION
assert int(frozen_hypotheses['lora_outcome_layer']['layer_index']) == L50
assert not frozen_hypotheses['probe_test_used_for_layer_selection']
assert stress_decision['gates']['l50_outcome_direction_positive_control']
assert not stress_decision['gates']['direct_torque_control_confirmed']

OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / f'qwen27b_postlora_l50_causal_subspace_decomposition_{RUN_VERSION}'
)
BLIND_V2_DATASET_DIR = OUTPUT_ROOT / 'blind_v2_dataset'
DEV_CLEAN_DIR = OUTPUT_ROOT / 'development_clean_states'
DEV_PATCH_DIR = OUTPUT_ROOT / 'development_patches'
TEST_CLEAN_DIR = OUTPUT_ROOT / 'confirmatory_clean_states'
TEST_PATCH_DIR = OUTPUT_ROOT / 'confirmatory_patches'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for directory in (
    OUTPUT_ROOT, BLIND_V2_DATASET_DIR, DEV_CLEAN_DIR, DEV_PATCH_DIR,
    TEST_CLEAN_DIR, TEST_PATCH_DIR, ANALYSIS_DIR, FIGURE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print('GPU:', GPU_NAME, '| memory GiB:', round(GPU_MEMORY_GIB, 1))
print('L50 full-state hypothesis: retained')
print('Direct torque-axis hypothesis: previously rejected')
print('Output:', OUTPUT_ROOT)


In [ ]:
%%cell_progress 4/16 Prepare Dataset v1 and freeze every exclusion source
cell_stage('Locating the original Dataset v1')
REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
SOURCE_DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
DRIVE_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(dataset_path):
    return dataset_path.is_dir() and all(
        (dataset_path / filename).is_file()
        for filename in REQUIRED_DATASET_FILES
    )

if not dataset_complete(SOURCE_DATASET_DIR):
    SOURCE_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            extraction_root = SOURCE_DATASET_DIR.resolve()
            for member in tqdm(
                [item for item in bundle.infolist() if not item.is_dir()],
                desc='Extracting Dataset v1', unit='file',
            ):
                target = (SOURCE_DATASET_DIR / member.filename).resolve()
                if extraction_root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, SOURCE_DATASET_DIR)
        if not dataset_complete(SOURCE_DATASET_DIR):
            nested = [
                item for item in SOURCE_DATASET_DIR.iterdir()
                if dataset_complete(item)
            ]
            assert len(nested) == 1, nested
            SOURCE_DATASET_DIR = nested[0]
    elif dataset_complete(DRIVE_DATASET_DIR):
        shutil.copytree(DRIVE_DATASET_DIR, SOURCE_DATASET_DIR, dirs_exist_ok=True)
    else:
        raise FileNotFoundError(
            'Place dataset_v1.zip or a complete dataset_v1 directory '
            'under VisualTorque/data on Drive.'
        )
assert dataset_complete(SOURCE_DATASET_DIR)

source_scenes = read_jsonl(SOURCE_DATASET_DIR / 'scenes.jsonl')
blind_v1_scenes = read_jsonl(BLIND_V1_DATASET_DIR / 'scenes.jsonl')
capture_shards = sorted(LORA_CAPTURE_DIR.glob('shard_*.pt'))
assert capture_shards

protocol_identity = {
    'schema_version': 'qwen27b-postlora-l50-causal-subspace-1.0',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'adapter_manifest_sha256': sha256_file(ADAPTER_MANIFEST_PATH),
    'frozen_hypotheses_sha256': sha256_file(FROZEN_HYPOTHESES_PATH),
    'stress_decision_sha256': sha256_file(STRESS_DECISION_PATH),
    'source_dataset_hashes': {
        name: sha256_file(SOURCE_DATASET_DIR / name)
        for name in sorted(REQUIRED_DATASET_FILES)
    },
    'blind_v1_scenes_sha256': sha256_file(BLIND_V1_DATASET_DIR / 'scenes.jsonl'),
    'lora_capture_config_sha256': sha256_file(
        LORA_CAPTURE_DIR / 'capture_config.json'
    ),
    'lora_capture_shard_hashes': {
        path.name: sha256_file(path) for path in capture_shards
    },
    'l50_layer': L50,
    'seed': SEED,
    'candidate_count': CANDIDATE_COUNT,
    'scenes_per_conflict_cell': SCENES_PER_CONFLICT_CELL,
    'dev_scenes_per_cell': DEV_SCENES_PER_CELL,
    'test_scenes_per_cell': TEST_SCENES_PER_CELL,
    'joint_ranks': list(JOINT_RANKS),
    'shared_ranks': list(SHARED_RANKS),
    'mapping_ranks': list(MAPPING_RANKS),
    'random_subspace_count': RANDOM_SUBSPACE_COUNT,
    'noninferiority_fraction': NONINFERIORITY_FRACTION,
    'selection_rule': (
        'among candidates with positive development sufficiency and necessity '
        'and >=80% mean recovery of both full-patch effects, choose the smallest '
        'rank; tie-break by larger minimum recovery then candidate name. If no '
        'candidate qualifies, choose maximum minimum recovery then smallest rank.'
    ),
    'primary_unit': 'physical scene after A/B/C permutation averaging',
    'confirmatory_read_rule': (
        'no confirmatory model forward before FROZEN_SUBSPACE_SELECTION.json exists'
    ),
    'confirmatory_gate_rule': (
        'full patch replicates; selected component has positive sufficiency and '
        'necessity; both are >=80% non-inferior to full; selected raw effects '
        'and effect-per-component-norm each beat 32 matched-rank random '
        'subspaces at empirical one-sided p<=.05; all A/B/C strata are positive'
    ),
}
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'frozen_protocol.json'
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Existing protocol differs. Increment RUN_VERSION.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

print('Dataset v1 scenes excluded:', len(source_scenes))
print('Blind v1 scenes excluded:', len(blind_v1_scenes))
print('Discovery capture shards frozen:', len(capture_shards))


In [ ]:
%%cell_progress 5/16 Generate and lock 48 entirely new conflict scenes
cell_stage('Enumerating new multi-stack candidates')
CONFLICT_CELLS = (('left', 'right'), ('right', 'left'))

def object_signature(scene):
    left = tuple(sorted(
        (int(stack.distance), int(stack.units))
        for stack in scene.stacks if stack.side.value == 'left'
    ))
    right = tuple(sorted(
        (int(stack.distance), int(stack.units))
        for stack in scene.stacks if stack.side.value == 'right'
    ))
    return left, right

def record_signature(scene):
    left = tuple(sorted(
        (int(stack['distance']), int(stack['units']))
        for stack in scene['stacks'] if stack['side'] == 'left'
    ))
    right = tuple(sorted(
        (int(stack['distance']), int(stack['units']))
        for stack in scene['stacks'] if stack['side'] == 'right'
    ))
    return left, right

def weight_cue_object(scene):
    if scene.left_units > scene.right_units:
        return 'left'
    if scene.left_units < scene.right_units:
        return 'right'
    return 'balanced'

excluded_signatures = {
    record_signature(scene) for scene in (*source_scenes, *blind_v1_scenes)
}
candidate_config = GenerationConfig(
    max_units=5, max_distance=6, multi_max_units=4,
    multi_scene_count=CANDIDATE_COUNT, seed=SEED,
)
candidates = generate_multi_stack_scenes(candidate_config)
pools = defaultdict(list)
for scene in candidates:
    signature = object_signature(scene)
    mirror = (signature[1], signature[0])
    if signature in excluded_signatures or mirror in excluded_signatures:
        continue
    key = (weight_cue_object(scene), scene.outcome.value)
    if key in CONFLICT_CELLS:
        pools[key].append(scene)
for cell in CONFLICT_CELLS:
    pools[cell].sort(key=lambda scene: stable_key(('pool', scene.scene_id)))
    assert len(pools[cell]) >= SCENES_PER_CONFLICT_CELL, (cell, len(pools[cell]))

cell_stage('Selecting mirror-disjoint scenes and balanced styles')
selected_scenes = []
selection_rows = []
selected_signatures = set()
style_names = ('cobalt_shift', 'violet_shift')
for cell in CONFLICT_CELLS:
    chosen = []
    for scene in pools[cell]:
        signature = object_signature(scene)
        mirror = (signature[1], signature[0])
        if signature in selected_signatures or mirror in selected_signatures:
            continue
        chosen.append(scene)
        selected_signatures.add(signature)
        if len(chosen) == SCENES_PER_CONFLICT_CELL:
            break
    assert len(chosen) == SCENES_PER_CONFLICT_CELL
    for cell_index, scene in enumerate(chosen):
        style = style_names[cell_index % 2]
        selection_rows.append({
            'scene_id': scene.scene_id,
            'weight_cue': cell[0],
            'outcome': cell[1],
            'style': style,
            'signature': object_signature(scene),
        })
        selected_scenes.append(scene)

selection_by_scene = {row['scene_id']: row for row in selection_rows}
for cell in CONFLICT_CELLS:
    for style in style_names:
        group = [
            row for row in selection_rows
            if (row['weight_cue'], row['outcome']) == cell
            and row['style'] == style
        ]
        group.sort(key=lambda row: stable_key(('split', row['scene_id'])))
        assert len(group) == 12
        for index, row in enumerate(group):
            row['split'] = (
                'subspace_development' if index < 6
                else 'subspace_confirmatory'
            )

assert Counter(row['split'] for row in selection_rows) == {
    'subspace_development': 24, 'subspace_confirmatory': 24
}
assert not selected_signatures & excluded_signatures
assert not {(right, left) for left, right in selected_signatures} & excluded_signatures

cell_stage('Rendering two new visual styles')
STYLES = {
    'cobalt_shift': RenderStyle(
        width=896, height=448, ticks_per_side=6, beam_y=286,
        block_size=26, block_gap=4,
        background=(250, 247, 234), ink=(30, 43, 68),
        block_fill=(48, 96, 181),
    ),
    'violet_shift': RenderStyle(
        width=1024, height=520, ticks_per_side=6, beam_y=330,
        block_size=30, block_gap=2,
        background=(242, 246, 251), ink=(51, 42, 67),
        block_fill=(137, 82, 171),
    ),
}
scene_records = []
question_records = []
for scene in tqdm(selected_scenes, desc='Rendering blind-v2 images', unit='image'):
    selection = selection_by_scene[scene.scene_id]
    relative_image = Path('images') / selection['style'] / f'{scene.scene_id}.png'
    destination = BLIND_V2_DATASET_DIR / relative_image
    if not destination.is_file():
        render_scene(scene, destination, STYLES[selection['style']])
    record = scene.to_record()
    record['split'] = selection['split']
    record['image_path'] = relative_image.as_posix()
    record['metadata'] = {
        **record.get('metadata', {}),
        'family': 'multi_stack',
        'blind_style': selection['style'],
        'blind_seed': str(SEED),
    }
    scene_records.append(record)
    for question in generate_questions(scene):
        if question.kind is QuestionKind.OUTCOME:
            question_records.append(question.to_record(split=selection['split']))

assert len(scene_records) == 48
assert len(question_records) == 48 * 6
write_jsonl_atomic(BLIND_V2_DATASET_DIR / 'scenes.jsonl', scene_records)
write_jsonl_atomic(BLIND_V2_DATASET_DIR / 'questions.jsonl', question_records)
write_jsonl_atomic(BLIND_V2_DATASET_DIR / 'counterfactual_pairs.jsonl', [])
write_json_atomic(BLIND_V2_DATASET_DIR / 'dataset_config.json', {
    'schema_version': 'visualtorque-blind-v2-subspace-dataset-1.0',
    'candidate_generation': candidate_config.to_record(),
    'excludes_dataset_v1_and_blind_v1_with_mirrors': True,
    'styles': {
        name: {
            key: getattr(style, key)
            for key in (
                'width', 'height', 'ticks_per_side', 'beam_y',
                'block_size', 'block_gap', 'background', 'ink', 'block_fill',
            )
        } for name, style in STYLES.items()
    },
})
write_json_atomic(BLIND_V2_DATASET_DIR / 'splits.json', {
    split: sorted(row['scene_id'] for row in selection_rows if row['split'] == split)
    for split in ('subspace_development', 'subspace_confirmatory')
})
write_json_atomic(BLIND_V2_DATASET_DIR / 'statistics.json', {
    'scene_count': 48,
    'question_count': len(question_records),
    'split_counts': dict(Counter(row['split'] for row in selection_rows)),
    'cell_style_split_counts': {
        '|'.join(key): int(value)
        for key, value in Counter(
            (row['weight_cue'], row['outcome'], row['style'], row['split'])
            for row in selection_rows
        ).items()
    },
})
write_json_atomic(BLIND_V2_DATASET_DIR / 'image_hashes.json', {
    row['scene_id']: sha256_file(
        BLIND_V2_DATASET_DIR / next(
            scene['image_path'] for scene in scene_records
            if scene['scene_id'] == row['scene_id']
        )
    ) for row in selection_rows
})
write_jsonl_atomic(OUTPUT_ROOT / 'blind_v2_scene_manifest.jsonl', selection_rows)

dataset_identity = {
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'dataset_hashes': {
        name: sha256_file(BLIND_V2_DATASET_DIR / name)
        for name in sorted(REQUIRED_DATASET_FILES)
    },
    'image_hashes_sha256': sha256_file(BLIND_V2_DATASET_DIR / 'image_hashes.json'),
    'scene_manifest_sha256': sha256_file(OUTPUT_ROOT / 'blind_v2_scene_manifest.jsonl'),
    'source_or_prior_geometry_overlap': 0,
    'source_or_prior_mirror_overlap': 0,
}
DATASET_LOCK_PATH = OUTPUT_ROOT / 'blind_v2_dataset_lock.json'
if DATASET_LOCK_PATH.is_file():
    assert json.loads(DATASET_LOCK_PATH.read_text(encoding='utf-8'))['identity'] == dataset_identity
else:
    write_json_atomic(DATASET_LOCK_PATH, {
        'identity': dataset_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

coverage = pd.DataFrame(selection_rows)
display(pd.crosstab(
    [coverage['weight_cue'], coverage['outcome'], coverage['style']],
    coverage['split'],
))
print('Fresh blind-v2 scenes:', len(scene_records))
print('Overlap with Dataset v1 / blind v1 and mirrors: 0')


In [ ]:
%%cell_progress 6/16 Build A/B/C-balanced development and confirmatory manifests
cell_stage('Loading all outcome permutations without model scoring')
import itertools
config = BehaviorRunConfig(
    dataset_dir=BLIND_V2_DATASET_DIR,
    output_dir=OUTPUT_ROOT / '_unused_behavior_config',
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    splits=('subspace_development', 'subspace_confirmatory'),
    batch_size=1,
    seed=SEED,
)
all_examples = list(load_behavior_examples(config))
assert len(all_examples) == 48 * 6
examples_by_scene = defaultdict(list)
for example in all_examples:
    examples_by_scene[example.scene_id].append(example)
assert all(len(rows) == 6 for rows in examples_by_scene.values())

def choose_three(scene_id):
    by_gold = {}
    for gold_label in ('A', 'B', 'C'):
        candidates = [
            example for example in examples_by_scene[scene_id]
            if example.gold_label == gold_label
        ]
        assert len(candidates) == 2
        by_gold[gold_label] = candidates
    latin_triples = []
    for triple in itertools.product(*(by_gold[label] for label in ('A', 'B', 'C'))):
        counts = Counter(
            (label, str(semantic))
            for example in triple
            for label, semantic in example.label_to_semantic.items()
        )
        if len(counts) == 9 and set(counts.values()) == {1}:
            latin_triples.append(triple)
    assert len(latin_triples) == 2, (scene_id, len(latin_triples))
    return list(min(
        latin_triples,
        key=lambda triple: stable_key(tuple(
            example.example_id for example in triple
        )),
    ))

dev_scene_ids = {
    row['scene_id'] for row in selection_rows
    if row['split'] == 'subspace_development'
}
test_scene_ids = {
    row['scene_id'] for row in selection_rows
    if row['split'] == 'subspace_confirmatory'
}
assert len(dev_scene_ids) == len(test_scene_ids) == 24
assert not dev_scene_ids & test_scene_ids

dev_examples = [
    example
    for scene_id in sorted(dev_scene_ids, key=stable_key)
    for example in choose_three(scene_id)
]
test_examples = [
    example
    for scene_id in sorted(test_scene_ids, key=stable_key)
    for example in choose_three(scene_id)
]
assert len(dev_examples) == len(test_examples) == 72
assert Counter(example.gold_label for example in dev_examples) == {
    'A': 24, 'B': 24, 'C': 24
}
assert Counter(example.gold_label for example in test_examples) == {
    'A': 24, 'B': 24, 'C': 24
}

def manifest_rows(examples, role):
    return [{
        'role': role,
        'example_id': example.example_id,
        'question_id': example.question_id,
        'scene_id': example.scene_id,
        'gold_label': example.gold_label,
        'gold_semantic': str(example.gold_semantic),
        'weight_cue': selection_by_scene[example.scene_id]['weight_cue'],
        'outcome': selection_by_scene[example.scene_id]['outcome'],
        'style': selection_by_scene[example.scene_id]['style'],
    } for example in examples]

DEV_MANIFEST_PATH = OUTPUT_ROOT / 'development_manifest.jsonl'
TEST_MANIFEST_PATH = OUTPUT_ROOT / 'confirmatory_manifest.jsonl'
write_jsonl_atomic(DEV_MANIFEST_PATH, manifest_rows(dev_examples, 'development'))
write_jsonl_atomic(TEST_MANIFEST_PATH, manifest_rows(test_examples, 'confirmatory'))
print('Development prompts:', len(dev_examples))
print('Confirmatory prompts:', len(test_examples))
display(pd.crosstab(
    pd.DataFrame(manifest_rows(test_examples, 'confirmatory'))['gold_label'],
    pd.DataFrame(manifest_rows(test_examples, 'confirmatory'))['gold_semantic'],
))


## 2. Capture development states and learn candidate subspaces

Only development prompts are opened here.


In [ ]:
%%cell_progress 7/16 Load the pinned base model and verified LoRA adapter
cell_stage('Loading the pinned BF16 base checkpoint')
adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
    device_map='cuda:0',
)
adapter.model.eval()
assert not hasattr(adapter.model, 'peft_config')

cell_stage('Verifying every saved adapter file')
def hash_tree(directory):
    return {
        file_path.relative_to(directory).as_posix(): sha256_file(file_path)
        for file_path in sorted(Path(directory).rglob('*'))
        if file_path.is_file()
    }

assert hash_tree(ADAPTER_DIR) == adapter_manifest['adapter_files'], (
    'Saved LoRA files do not match ADAPTER_MANIFEST.json.'
)

cell_stage('Attaching LoRA read-only')
adapter.model = PeftModel.from_pretrained(
    adapter.model,
    str(ADAPTER_DIR),
    is_trainable=False,
)
adapter.model.eval()
for parameter in adapter.model.parameters():
    parameter.requires_grad_(False)
assert not any(parameter.requires_grad for parameter in adapter.model.parameters())

def discover_decoder_layers(model):
    candidates = [
        (name, module)
        for name, module in model.named_modules()
        if (
            isinstance(module, torch.nn.ModuleList)
            and len(module) == EXPECTED_LAYER_COUNT
            and name.endswith('language_model.layers')
        )
    ]
    if not candidates:
        candidates = [
            (name, module)
            for name, module in model.named_modules()
            if isinstance(module, torch.nn.ModuleList)
            and len(module) == EXPECTED_LAYER_COUNT
        ]
    assert candidates, 'Could not locate the 64 decoder layers.'
    candidates.sort(key=lambda item: (item[0].count('.'), len(item[0])))
    return candidates[0]

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
print('Model class:', type(adapter.model).__name__)
print('Decoder path:', LAYER_PATH)
print('Adapter verification: passed')


In [ ]:
%%cell_progress 8/16 Define clean capture and batched L50 state patching
cell_stage('Defining exact-score activation utilities')

@contextmanager
def model_mode(mode):
    if mode == 'native':
        with adapter.model.disable_adapter():
            yield
    elif mode == 'lora':
        yield
    else:
        raise ValueError(mode)

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        tensor = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        tensor = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        tensor = output.last_hidden_state
    else:
        raise TypeError(type(output).__name__)
    assert tensor.ndim == 3, tuple(tensor.shape)
    return tensor

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def safe_example_stem(example_id):
    return hashlib.sha256(example_id.encode('utf-8')).hexdigest()[:24]

def prepare_repeated(example, repeats=1):
    plan = adapter._candidate_plan(example)
    if plan is None:
        raise RuntimeError('A/B/C must be exact single-token candidates.')
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for _ in range(repeats)],
            add_generation_prompt=True,
        )),
        fixed_prefix,
    )
    mask = inputs['attention_mask'].bool()
    positions = (
        mask.long()
        * torch.arange(mask.shape[1], device=mask.device).unsqueeze(0)
    ).max(dim=1).values
    return inputs, positions, token_by_label

def score_logits(example, logits, positions, token_by_label, conditions):
    rows = torch.arange(logits.shape[0], device=logits.device)
    logprobs = torch.log_softmax(logits[rows, positions, :].float(), dim=-1)
    results = []
    for row_index, condition in enumerate(conditions):
        semantic_scores = {
            str(example.label_to_semantic[label]): float(
                logprobs[row_index, token_by_label[label]].detach().cpu()
            )
            for label in example.labels
        }
        prediction = max(semantic_scores, key=semantic_scores.get)
        outcome = str(example.gold_semantic)
        direction_sign = 1.0 if outcome == 'left' else -1.0
        results.append({
            'condition': condition,
            'example_id': example.example_id,
            'scene_id': example.scene_id,
            'variant': example.variant,
            'gold_label': example.gold_label,
            'outcome': outcome,
            'weight_cue': selection_by_scene[example.scene_id]['weight_cue'],
            'style': selection_by_scene[example.scene_id]['style'],
            'semantic_logprobs': semantic_scores,
            'predicted_semantic': prediction,
            'correct': prediction == outcome,
            'left_right_margin': float(
                semantic_scores['left'] - semantic_scores['right']
            ),
            'correct_margin': float(direction_sign * (
                semantic_scores['left'] - semantic_scores['right']
            )),
        })
    return results

def forward_clean(example, mode):
    inputs, positions, token_by_label = prepare_repeated(example, 1)
    captured = {}
    rows = torch.arange(1, device=positions.device)
    def capture_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        captured['state'] = hidden[
            rows, positions, :
        ].detach().cpu().to(torch.float16)
    handle = DECODER_LAYERS[L50].register_forward_hook(capture_hook)
    try:
        with model_mode(mode):
            with torch.inference_mode():
                output = adapter.model(**inputs, use_cache=False)
    finally:
        handle.remove()
    record = score_logits(
        example, output.logits, positions, token_by_label, [f'{mode}_clean']
    )[0]
    del inputs, output
    return record, captured['state'][0]

def forward_target_batch(example, recipient_mode, named_targets):
    named_targets = list(named_targets)
    inputs, positions, token_by_label = prepare_repeated(example, len(named_targets))
    rows = torch.arange(len(named_targets), device=positions.device)
    targets = torch.stack([
        state.detach().cpu().float() for _, state in named_targets
    ])

    def patch_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        modified = hidden.clone()
        modified[rows, positions, :] = targets.to(hidden.device, hidden.dtype)
        return replace_output_hidden(output, modified)

    handle = DECODER_LAYERS[L50].register_forward_hook(patch_hook)
    try:
        with model_mode(recipient_mode):
            with torch.inference_mode():
                output = adapter.model(**inputs, use_cache=False)
    finally:
        handle.remove()
    records = score_logits(
        example, output.logits, positions, token_by_label,
        [name for name, _ in named_targets],
    )
    del inputs, output
    return records

def chunks(values, size):
    values = list(values)
    for start in range(0, len(values), size):
        yield values[start:start + size]

def capture_clean_examples(examples, directory, role, allow_creation):
    identity = {
        'schema_version': 'qwen27b-postlora-l50-clean-capture-1.0',
        'role': role,
        'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
        'dataset_lock_sha256': sha256_file(DATASET_LOCK_PATH),
        'manifest_sha256': sha256_file(
            DEV_MANIFEST_PATH if role == 'development' else TEST_MANIFEST_PATH
        ),
        'layer_index': L50,
        'example_count': len(examples),
    }
    identity_path = Path(directory) / 'identity.json'
    if identity_path.is_file():
        assert json.loads(identity_path.read_text(encoding='utf-8')) == identity
    else:
        assert allow_creation
        write_json_atomic(identity_path, identity)
    index_rows = []
    for example in tqdm(examples, desc=f'{role} clean native/LoRA', unit='prompt'):
        file_path = Path(directory) / f'{safe_example_stem(example.example_id)}.pt'
        if not file_path.is_file():
            assert allow_creation
            native_record, native_state = forward_clean(example, 'native')
            lora_record, lora_state = forward_clean(example, 'lora')
            temporary = file_path.with_suffix('.pt.tmp')
            torch.save({
                'example_id': example.example_id,
                'native_record': native_record,
                'lora_record': lora_record,
                'native_state': native_state,
                'lora_state': lora_state,
            }, temporary)
            temporary.replace(file_path)
        payload = torch.load(file_path, map_location='cpu', weights_only=False)
        assert payload['example_id'] == example.example_id
        index_rows.append({'example_id': example.example_id, 'file': file_path.name})
    index_path = Path(directory) / 'index.jsonl'
    write_jsonl_atomic(index_path, index_rows)
    return index_rows

def load_clean_payload(directory, example):
    file_path = Path(directory) / f'{safe_example_stem(example.example_id)}.pt'
    payload = torch.load(file_path, map_location='cpu', weights_only=False)
    assert payload['example_id'] == example.example_id
    return payload

print('L50 clean-capture and state-patch utilities ready.')


In [ ]:
%%cell_progress 9/16 Capture development Native and LoRA L50 states
cell_stage('Capturing development clean states only')
if not (OUTPUT_ROOT / 'FROZEN_SUBSPACE_SELECTION.json').is_file():
    assert not any(TEST_CLEAN_DIR.glob('*.pt')), (
        'Confirmatory clean states must not exist before subspace selection.'
    )
dev_clean_index = capture_clean_examples(
    dev_examples, DEV_CLEAN_DIR, 'development', allow_creation=True
)
assert len(dev_clean_index) == len(dev_examples)

cell_stage('Checking self-patch parity')
smoke_example = dev_examples[0]
smoke_payload = load_clean_payload(DEV_CLEAN_DIR, smoke_example)
parity_rows = []
for mode in ('native', 'lora'):
    records = forward_target_batch(
        smoke_example, mode,
        [(f'{mode}_self_patch', smoke_payload[f'{mode}_state'])],
    )
    clean = smoke_payload[f'{mode}_record']
    patched = records[0]
    clean_scores = np.asarray([
        clean['semantic_logprobs'][side] for side in ('left', 'balanced', 'right')
    ])
    patch_scores = np.asarray([
        patched['semantic_logprobs'][side] for side in ('left', 'balanced', 'right')
    ])
    error = float(np.max(np.abs(
        (clean_scores - clean_scores.mean())
        - (patch_scores - patch_scores.mean())
    )))
    parity_rows.append({'mode': mode, 'centered_logprob_error': error})
    assert error < 0.03, (mode, error)
write_json_atomic(DEV_CLEAN_DIR / 'self_patch_parity.json', parity_rows)
display(pd.DataFrame(parity_rows))
print('Development clean prompts:', len(dev_clean_index))


In [ ]:
%%cell_progress 10/16 Learn joint, scene-shared, and mapping-residual L50 bases
cell_stage('Assembling development LoRA-minus-Native deltas')
dev_rows = []
dev_deltas = []
for example in dev_examples:
    payload = load_clean_payload(DEV_CLEAN_DIR, example)
    delta = payload['lora_state'].float() - payload['native_state'].float()
    dev_deltas.append(delta.numpy())
    dev_rows.append({
        'example_id': example.example_id,
        'scene_id': example.scene_id,
        'gold_label': example.gold_label,
        'outcome': str(example.gold_semantic),
    })
dev_deltas = np.stack(dev_deltas).astype(np.float32)
dev_metadata = pd.DataFrame(dev_rows)
HIDDEN_SIZE = int(dev_deltas.shape[1])
assert dev_deltas.shape[0] == 72

def row_basis(matrix, tolerance_ratio=1e-6):
    matrix = np.asarray(matrix, dtype=np.float32)
    _u, singular_values, vt = np.linalg.svd(matrix, full_matrices=False)
    threshold = max(float(singular_values[0]) * tolerance_ratio, 1e-8)
    rank = int(np.sum(singular_values > threshold))
    assert rank > 0
    basis = vt[:rank].astype(np.float32)
    assert np.max(np.abs(basis @ basis.T - np.eye(rank))) < 2e-4
    return basis, singular_values[:rank].astype(np.float32)

joint_basis, joint_singular = row_basis(dev_deltas)
scene_mean_by_id = {
    scene_id: dev_deltas[group.index.to_numpy()].mean(axis=0)
    for scene_id, group in dev_metadata.groupby('scene_id', sort=True)
}
shared_matrix = np.stack([
    scene_mean_by_id[scene_id] for scene_id in sorted(scene_mean_by_id)
])
shared_basis, shared_singular = row_basis(shared_matrix)
mapping_matrix = np.stack([
    delta - scene_mean_by_id[scene_id]
    for delta, scene_id in zip(dev_deltas, dev_metadata['scene_id'])
])
mapping_orth_matrix = mapping_matrix - (
    (mapping_matrix @ shared_basis.T) @ shared_basis
)
mapping_basis, mapping_singular = row_basis(mapping_orth_matrix)
assert np.max(np.abs(shared_basis @ mapping_basis.T)) < 2e-4

cell_stage('Building frozen outcome and L50 torque probe baselines')
probe_records = {
    row['target']: row for row in frozen_hypotheses['saved_probes']
}
outcome_probe_path = Path(probe_records['outcome']['path'])
assert sha256_file(outcome_probe_path) == probe_records['outcome']['sha256']
outcome_probe = joblib.load(outcome_probe_path)
outcome_scaler = outcome_probe.named_steps['standardscaler']
outcome_classifier = outcome_probe.named_steps['ridgeclassifier']
outcome_classes = [str(value) for value in outcome_classifier.classes_.tolist()]
left_index = outcome_classes.index('left')
right_index = outcome_classes.index('right')
outcome_axis = (
    np.asarray(outcome_classifier.coef_[left_index], dtype=np.float32)
    - np.asarray(outcome_classifier.coef_[right_index], dtype=np.float32)
) / np.asarray(outcome_scaler.scale_, dtype=np.float32)
outcome_axis /= np.linalg.norm(outcome_axis)

capture_config = json.loads(
    (LORA_CAPTURE_DIR / 'capture_config.json').read_text(encoding='utf-8')
)['identity']
capture_layers = [int(value) for value in capture_config['layer_indices']]
l50_position = capture_layers.index(L50)
probe_X = []
probe_y = []
for file_path in tqdm(capture_shards, desc='Loading L50 probe-train states', unit='shard'):
    payload = torch.load(file_path, map_location='cpu', weights_only=False)
    activations = payload['activations'].float().numpy()
    for row_index, record in enumerate(payload['records']):
        if 'main' not in str(record['role']) or record['split'] != 'probe_train':
            continue
        probe_X.append(activations[row_index, l50_position])
        probe_y.append(float(record['net_torque']))
probe_X = np.stack(probe_X).astype(np.float32)
probe_y = np.asarray(probe_y, dtype=np.float32)
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
torque_scaler = StandardScaler().fit(probe_X)
torque_model = Ridge(alpha=100.0, solver='lsqr', max_iter=5000).fit(
    torque_scaler.transform(probe_X), probe_y
)
torque_axis = np.asarray(torque_model.coef_, dtype=np.float32) / np.asarray(
    torque_scaler.scale_, dtype=np.float32
)
torque_axis /= np.linalg.norm(torque_axis)
if np.corrcoef(probe_X @ torque_axis, probe_y)[0, 1] < 0:
    torque_axis *= -1

probe_span, _ = np.linalg.qr(np.stack([outcome_axis, torque_axis], axis=1))
probe_span = probe_span.T.astype(np.float32)

def available_ranks(requested, available):
    # "Low rank" candidates must leave at least one learned delta direction
    # outside the candidate so matched-rank random controls remain identifiable.
    return tuple(rank for rank in requested if rank < available)

joint_ranks = available_ranks(JOINT_RANKS, joint_basis.shape[0])
shared_ranks = available_ranks(SHARED_RANKS, shared_basis.shape[0])
mapping_ranks = available_ranks(MAPPING_RANKS, mapping_basis.shape[0])
assert joint_ranks and shared_ranks and mapping_ranks

component_specs = []
for family, basis, ranks in (
    ('joint_svd', joint_basis, joint_ranks),
    ('scene_shared', shared_basis, shared_ranks),
    ('mapping_residual', mapping_basis, mapping_ranks),
):
    for rank in ranks:
        component_specs.append({
            'name': f'{family}_rank{rank}',
            'family': family,
            'rank': int(rank),
            'basis': torch.from_numpy(basis[:rank]),
        })
component_specs.extend([
    {
        'name': 'outcome_probe_rank1', 'family': 'probe_outcome', 'rank': 1,
        'basis': torch.from_numpy(outcome_axis[None, :]),
    },
    {
        'name': 'torque_probe_rank1', 'family': 'probe_torque', 'rank': 1,
        'basis': torch.from_numpy(torque_axis[None, :]),
    },
    {
        'name': 'probe_span_rank2', 'family': 'probe_span', 'rank': 2,
        'basis': torch.from_numpy(probe_span),
    },
])

SUBSPACE_PATH = OUTPUT_ROOT / 'development_frozen_subspaces.pt'
subspace_payload = {
    'schema_version': 'qwen27b-postlora-l50-development-subspaces-1.0',
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'development_clean_identity_sha256': sha256_file(DEV_CLEAN_DIR / 'identity.json'),
    'hidden_size': HIDDEN_SIZE,
    'joint_basis': torch.from_numpy(joint_basis),
    'joint_singular_values': torch.from_numpy(joint_singular),
    'shared_basis': torch.from_numpy(shared_basis),
    'shared_singular_values': torch.from_numpy(shared_singular),
    'mapping_basis': torch.from_numpy(mapping_basis),
    'mapping_singular_values': torch.from_numpy(mapping_singular),
    'outcome_axis': torch.from_numpy(outcome_axis),
    'torque_axis_l50': torch.from_numpy(torque_axis),
    'probe_span': torch.from_numpy(probe_span),
    'component_manifest': [
        {key: value for key, value in spec.items() if key != 'basis'}
        for spec in component_specs
    ],
}
if SUBSPACE_PATH.is_file():
    previous = torch.load(SUBSPACE_PATH, map_location='cpu', weights_only=False)
    assert previous['protocol_sha256'] == subspace_payload['protocol_sha256']
    for key in ('joint_basis', 'shared_basis', 'mapping_basis'):
        assert torch.allclose(previous[key], subspace_payload[key], atol=1e-6, rtol=0)
else:
    torch.save(subspace_payload, SUBSPACE_PATH)

subspace_summary = pd.DataFrame([
    {'family': 'joint_svd', 'available_rank': joint_basis.shape[0]},
    {'family': 'scene_shared', 'available_rank': shared_basis.shape[0]},
    {'family': 'mapping_residual', 'available_rank': mapping_basis.shape[0]},
])
display(subspace_summary)
print('Candidate projected subspaces:', len(component_specs))
print('Hidden size:', HIDDEN_SIZE)


In [ ]:
%%cell_progress 11/16 Run or resume development subspace patches
cell_stage('Checking development-patch identity')
dev_patch_identity = {
    'schema_version': 'qwen27b-postlora-l50-development-patches-1.0',
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'dataset_lock_sha256': sha256_file(DATASET_LOCK_PATH),
    'manifest_sha256': sha256_file(DEV_MANIFEST_PATH),
    'subspaces_sha256': sha256_file(SUBSPACE_PATH),
    'component_names': ['full_delta'] + [spec['name'] for spec in component_specs],
    'recipients': ['native_sufficiency', 'lora_necessity'],
}
DEV_PATCH_IDENTITY_PATH = DEV_PATCH_DIR / 'identity.json'
if DEV_PATCH_IDENTITY_PATH.is_file():
    assert json.loads(
        DEV_PATCH_IDENTITY_PATH.read_text(encoding='utf-8')
    ) == dev_patch_identity
else:
    write_json_atomic(DEV_PATCH_IDENTITY_PATH, dev_patch_identity)

def project_delta(delta, basis):
    delta = delta.detach().cpu().float()
    basis = basis.detach().cpu().float()
    return (delta @ basis.T) @ basis

def components_for_delta(delta, specs):
    values = [('full_delta', delta.detach().cpu().float(), 'full_delta', HIDDEN_SIZE)]
    for spec in specs:
        values.append((
            spec['name'], project_delta(delta, spec['basis']),
            spec['family'], int(spec['rank']),
        ))
    return values

completed = {
    path.stem for path in DEV_PATCH_DIR.glob('*.json')
    if path.name != 'identity.json'
}
for example in tqdm(dev_examples, desc='Development L50 patches', unit='prompt'):
    safe_id = safe_example_stem(example.example_id)
    output_path = DEV_PATCH_DIR / f'{safe_id}.json'
    if output_path.stem in completed:
        continue
    clean = load_clean_payload(DEV_CLEAN_DIR, example)
    native_state = clean['native_state'].float()
    lora_state = clean['lora_state'].float()
    delta = lora_state - native_state
    delta_norm = float(torch.linalg.vector_norm(delta))
    components = components_for_delta(delta, component_specs)
    records = []
    for recipient_mode, sign, clean_state, role in (
        ('native', 1.0, native_state, 'native_sufficiency'),
        ('lora', -1.0, lora_state, 'lora_necessity'),
    ):
        for batch in chunks(components, PATCH_BATCH_SIZE):
            named_targets = [
                (f'{role}|{name}', clean_state + sign * component)
                for name, component, _family, _rank in batch
            ]
            batch_records = forward_target_batch(
                example, recipient_mode, named_targets
            )
            for record, (name, component, family, rank) in zip(batch_records, batch):
                record.update({
                    'recipient_role': role,
                    'component_name': name,
                    'component_family': family,
                    'component_rank': int(rank),
                    'component_norm': float(torch.linalg.vector_norm(component)),
                    'delta_norm': delta_norm,
                    'component_norm_fraction': float(
                        torch.linalg.vector_norm(component) / max(delta_norm, 1e-12)
                    ),
                })
            records.extend(batch_records)
    assert len(records) == 2 * len(components)
    write_json_atomic(output_path, {
        'schema_version': 'qwen27b-postlora-l50-development-patch-records-1.0',
        'example_id': example.example_id,
        'records': records,
    })
print('Development patch prompts complete:', len(dev_examples))


In [ ]:
%%cell_progress 12/16 Select and freeze the minimum development-qualified subspace
cell_stage('Loading development patches and clean baselines')
patch_rows = []
for path in sorted(DEV_PATCH_DIR.glob('*.json')):
    if path.name == 'identity.json':
        continue
    patch_rows.extend(json.loads(path.read_text(encoding='utf-8'))['records'])
dev_patch_frame = pd.DataFrame(patch_rows)
assert dev_patch_frame['example_id'].nunique() == len(dev_examples)

clean_rows = []
for example in dev_examples:
    payload = load_clean_payload(DEV_CLEAN_DIR, example)
    for mode in ('native', 'lora'):
        record = dict(payload[f'{mode}_record'])
        record['mode'] = mode
        clean_rows.append(record)
dev_clean_frame = pd.DataFrame(clean_rows)
clean_wide = dev_clean_frame.pivot(
    index='example_id', columns='mode', values=['correct_margin', 'correct']
)
clean_wide.columns = ['_'.join(values) for values in clean_wide.columns]
clean_wide = clean_wide.reset_index()

prompt_effect = dev_patch_frame.merge(clean_wide, on='example_id', validate='many_to_one')
prompt_effect['effect'] = np.where(
    prompt_effect['recipient_role'] == 'native_sufficiency',
    prompt_effect['correct_margin'] - prompt_effect['correct_margin_native'],
    prompt_effect['correct_margin_lora'] - prompt_effect['correct_margin'],
)
prompt_effect['accuracy_effect'] = np.where(
    prompt_effect['recipient_role'] == 'native_sufficiency',
    prompt_effect['correct'].astype(float) - prompt_effect['correct_native'].astype(float),
    prompt_effect['correct_lora'].astype(float) - prompt_effect['correct'].astype(float),
)
dev_scene_effect = (
    prompt_effect.groupby([
        'scene_id', 'recipient_role', 'component_name',
        'component_family', 'component_rank',
    ], as_index=False).agg(
        effect=('effect', 'mean'),
        accuracy_effect=('accuracy_effect', 'mean'),
        component_norm_fraction=('component_norm_fraction', 'mean'),
    )
)
dev_scene_effect.to_csv(ANALYSIS_DIR / 'development_scene_effects.csv', index=False)

def bootstrap_summary(values, seed):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    assert len(values)
    rng = np.random.default_rng(seed)
    draws = np.empty(BOOTSTRAP_REPLICATES, dtype=float)
    for index in range(BOOTSTRAP_REPLICATES):
        sampled = values[rng.integers(0, len(values), size=len(values))]
        draws[index] = float(sampled.mean())
    low, high = np.quantile(draws, [0.025, 0.975])
    signs = rng.choice((-1.0, 1.0), size=(SIGN_FLIP_REPLICATES, len(values)))
    null = np.mean(signs * values[None, :], axis=1)
    p_value = float(
        (np.sum(np.abs(null) >= abs(values.mean())) + 1)
        / (SIGN_FLIP_REPLICATES + 1)
    )
    return {
        'n_scenes': int(len(values)),
        'mean': float(values.mean()),
        'ci_low': float(low),
        'ci_high': float(high),
        'sign_flip_p': p_value,
    }

full_by_role = {
    role: dev_scene_effect[
        (dev_scene_effect['recipient_role'] == role)
        & (dev_scene_effect['component_name'] == 'full_delta')
    ].set_index('scene_id')['effect']
    for role in ('native_sufficiency', 'lora_necessity')
}
assert all(len(values) == 24 for values in full_by_role.values())
full_patch_development_positive = all(
    values.mean() > 0 for values in full_by_role.values()
)

candidate_rows = []
for candidate_index, (component_name, group) in enumerate(
    dev_scene_effect[dev_scene_effect['component_name'] != 'full_delta']
    .groupby('component_name', sort=True)
):
    row = {
        'component_name': component_name,
        'component_family': group['component_family'].iloc[0],
        'component_rank': int(group['component_rank'].iloc[0]),
        'component_norm_fraction': float(group['component_norm_fraction'].mean()),
    }
    recoveries = []
    qualified = bool(full_patch_development_positive)
    for role_index, role in enumerate(('native_sufficiency', 'lora_necessity')):
        current = group[group['recipient_role'] == role].set_index('scene_id')['effect']
        full = full_by_role[role].reindex(current.index)
        summary = bootstrap_summary(current, SEED + 3000 + candidate_index * 10 + role_index)
        noninferiority = bootstrap_summary(
            current - NONINFERIORITY_FRACTION * full,
            SEED + 4000 + candidate_index * 10 + role_index,
        )
        recovery = float(current.mean() / max(abs(full.mean()), 1e-8))
        recoveries.append(recovery)
        prefix = 'sufficiency' if role == 'native_sufficiency' else 'necessity'
        row.update({
            f'{prefix}_mean': summary['mean'],
            f'{prefix}_ci_low': summary['ci_low'],
            f'{prefix}_ci_high': summary['ci_high'],
            f'{prefix}_recovery': recovery,
            f'{prefix}_noninferiority_mean': noninferiority['mean'],
            f'{prefix}_noninferiority_ci_low': noninferiority['ci_low'],
        })
        qualified = qualified and summary['ci_low'] > 0 and recovery >= NONINFERIORITY_FRACTION
    row['minimum_recovery'] = float(min(recoveries))
    row['development_qualified'] = bool(qualified)
    candidate_rows.append(row)

candidate_summary = pd.DataFrame(candidate_rows)
qualified = candidate_summary[candidate_summary['development_qualified']]
if len(qualified):
    selected_row = qualified.sort_values(
        ['component_rank', 'minimum_recovery', 'component_name'],
        ascending=[True, False, True],
    ).iloc[0]
    selection_status = 'qualified_minimum_rank'
else:
    selected_row = candidate_summary.sort_values(
        ['minimum_recovery', 'component_rank', 'component_name'],
        ascending=[False, True, True],
    ).iloc[0]
    selection_status = 'no_candidate_qualified_fallback'

candidate_summary = candidate_summary.sort_values(
    ['development_qualified', 'component_rank', 'minimum_recovery'],
    ascending=[False, True, False],
).reset_index(drop=True)
candidate_summary.to_csv(ANALYSIS_DIR / 'development_candidate_summary.csv', index=False)
selected = selected_row.to_dict()

SELECTION_PATH = OUTPUT_ROOT / 'FROZEN_SUBSPACE_SELECTION.json'
selection_payload = {
    'schema_version': 'qwen27b-postlora-l50-frozen-subspace-selection-1.0',
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'subspaces_sha256': sha256_file(SUBSPACE_PATH),
    'development_patch_identity_sha256': sha256_file(DEV_PATCH_IDENTITY_PATH),
    'selection_rule': protocol_identity['selection_rule'],
    'selection_status': selection_status,
    'full_patch_development_positive': full_patch_development_positive,
    'selected_candidate': selected,
    'selected_at_utc': datetime.now(timezone.utc).isoformat(),
    'confirmatory_model_outputs_read_before_selection': False,
}
if SELECTION_PATH.is_file():
    previous = json.loads(SELECTION_PATH.read_text(encoding='utf-8'))
    assert previous['selected_candidate']['component_name'] == selected['component_name']
    selection_payload = previous
else:
    assert not any(TEST_CLEAN_DIR.glob('*.pt'))
    assert not any(TEST_PATCH_DIR.glob('*.json'))
    write_json_atomic(SELECTION_PATH, selection_payload)

selected = selection_payload['selected_candidate']
SELECTED_NAME = str(selected['component_name'])
SELECTED_FAMILY = str(selected['component_family'])
SELECTED_RANK = int(selected['component_rank'])
selected_spec = next(spec for spec in component_specs if spec['name'] == SELECTED_NAME)
selected_basis = selected_spec['basis'].float()

display(candidate_summary.head(15).round(4))
print('Selection status:', selection_payload['selection_status'])
print('Frozen candidate:', SELECTED_NAME)


## 3. Confirm one frozen subspace on untouched scenes

The confirmatory states are not created until the selection file exists.


In [ ]:
%%cell_progress 13/16 Capture confirmatory states and freeze matched random subspaces
cell_stage('Capturing confirmatory clean states after selection freeze')
assert SELECTION_PATH.is_file()
test_clean_index = capture_clean_examples(
    test_examples, TEST_CLEAN_DIR, 'confirmatory', allow_creation=True
)
assert len(test_clean_index) == len(test_examples)

cell_stage('Sampling same-rank controls inside the development delta span')
joint_parent = torch.from_numpy(joint_basis).float()
assert SELECTED_RANK < joint_parent.shape[0], (
    'Random same-rank controls require a larger development delta span.',
    SELECTED_RANK, joint_parent.shape[0],
)
rng = np.random.default_rng(SEED + 5050 + SELECTED_RANK)
random_bases = []
for random_index in range(RANDOM_SUBSPACE_COUNT):
    coefficients = rng.normal(
        size=(joint_parent.shape[0], SELECTED_RANK)
    ).astype(np.float32)
    q, _ = np.linalg.qr(coefficients)
    basis = q[:, :SELECTED_RANK].T @ joint_parent.numpy()
    basis = basis.astype(np.float32)
    assert np.max(np.abs(
        basis @ basis.T - np.eye(SELECTED_RANK)
    )) < 3e-4
    random_bases.append(basis)

RANDOM_SUBSPACE_PATH = OUTPUT_ROOT / 'frozen_random_subspaces.pt'
random_payload = {
    'schema_version': 'qwen27b-postlora-l50-random-subspaces-1.0',
    'selection_sha256': sha256_file(SELECTION_PATH),
    'parent': 'development_joint_delta_row_span',
    'parent_rank': int(joint_parent.shape[0]),
    'subspace_rank': SELECTED_RANK,
    'count': RANDOM_SUBSPACE_COUNT,
    'seed': SEED + 5050 + SELECTED_RANK,
    'bases': torch.from_numpy(np.stack(random_bases)),
}
if RANDOM_SUBSPACE_PATH.is_file():
    previous = torch.load(
        RANDOM_SUBSPACE_PATH, map_location='cpu', weights_only=False
    )
    assert torch.allclose(
        previous['bases'], random_payload['bases'], atol=1e-6, rtol=0
    )
else:
    torch.save(random_payload, RANDOM_SUBSPACE_PATH)
print('Confirmatory clean prompts:', len(test_clean_index))
print('Random controls:', RANDOM_SUBSPACE_COUNT, 'rank:', SELECTED_RANK)


In [ ]:
%%cell_progress 14/16 Run or resume confirmatory full, projected, and control patches
cell_stage('Freezing the confirmatory patch inventory')
diagnostic_specs = {
    'scene_shared_full': torch.from_numpy(shared_basis).float(),
    'mapping_residual_full': torch.from_numpy(mapping_basis).float(),
    'outcome_probe_rank1': torch.from_numpy(outcome_axis[None, :]).float(),
    'torque_probe_rank1': torch.from_numpy(torque_axis[None, :]).float(),
    'probe_span_rank2': torch.from_numpy(probe_span).float(),
}
diagnostic_specs = {
    name: basis for name, basis in diagnostic_specs.items()
    if name != SELECTED_NAME
}
test_component_names = [
    'full_delta', SELECTED_NAME, 'selected_orthogonal_complement',
    *diagnostic_specs.keys(),
    *[f'random_subspace_{index:02d}' for index in range(RANDOM_SUBSPACE_COUNT)],
]
assert len(test_component_names) == len(set(test_component_names))
test_patch_identity = {
    'schema_version': 'qwen27b-postlora-l50-confirmatory-patches-1.0',
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'dataset_lock_sha256': sha256_file(DATASET_LOCK_PATH),
    'manifest_sha256': sha256_file(TEST_MANIFEST_PATH),
    'selection_sha256': sha256_file(SELECTION_PATH),
    'random_subspaces_sha256': sha256_file(RANDOM_SUBSPACE_PATH),
    'components': test_component_names,
    'recipients': ['native_sufficiency', 'lora_necessity'],
}
TEST_PATCH_IDENTITY_PATH = TEST_PATCH_DIR / 'identity.json'
if TEST_PATCH_IDENTITY_PATH.is_file():
    assert json.loads(
        TEST_PATCH_IDENTITY_PATH.read_text(encoding='utf-8')
    ) == test_patch_identity
else:
    write_json_atomic(TEST_PATCH_IDENTITY_PATH, test_patch_identity)

random_bases_tensor = random_payload['bases'].float()
completed = {
    path.stem for path in TEST_PATCH_DIR.glob('*.json')
    if path.name != 'identity.json'
}
for example in tqdm(test_examples, desc='Confirmatory L50 patches', unit='prompt'):
    safe_id = safe_example_stem(example.example_id)
    output_path = TEST_PATCH_DIR / f'{safe_id}.json'
    if output_path.stem in completed:
        continue
    clean = load_clean_payload(TEST_CLEAN_DIR, example)
    native_state = clean['native_state'].float()
    lora_state = clean['lora_state'].float()
    delta = lora_state - native_state
    delta_norm = float(torch.linalg.vector_norm(delta))
    selected_component = project_delta(delta, selected_basis)
    components = [
        ('full_delta', delta, 'full_delta', HIDDEN_SIZE),
        (SELECTED_NAME, selected_component, SELECTED_FAMILY, SELECTED_RANK),
        (
            'selected_orthogonal_complement', delta - selected_component,
            'selected_complement', HIDDEN_SIZE - SELECTED_RANK,
        ),
    ]
    for name, basis in diagnostic_specs.items():
        components.append((
            name, project_delta(delta, basis),
            'diagnostic', int(basis.shape[0]),
        ))
    for random_index in range(RANDOM_SUBSPACE_COUNT):
        basis = random_bases_tensor[random_index]
        components.append((
            f'random_subspace_{random_index:02d}',
            project_delta(delta, basis),
            'random_subspace', SELECTED_RANK,
        ))
    assert [value[0] for value in components] == test_component_names

    records = []
    for recipient_mode, sign, clean_state, role in (
        ('native', 1.0, native_state, 'native_sufficiency'),
        ('lora', -1.0, lora_state, 'lora_necessity'),
    ):
        for batch in chunks(components, PATCH_BATCH_SIZE):
            named_targets = [
                (f'{role}|{name}', clean_state + sign * component)
                for name, component, _family, _rank in batch
            ]
            batch_records = forward_target_batch(
                example, recipient_mode, named_targets
            )
            for record, (name, component, family, rank) in zip(batch_records, batch):
                component_norm = float(torch.linalg.vector_norm(component))
                record.update({
                    'recipient_role': role,
                    'component_name': name,
                    'component_family': family,
                    'component_rank': int(rank),
                    'component_norm': component_norm,
                    'delta_norm': delta_norm,
                    'component_norm_fraction': component_norm / max(delta_norm, 1e-12),
                    'component_energy_fraction': (
                        component_norm / max(delta_norm, 1e-12)
                    ) ** 2,
                })
            records.extend(batch_records)
    assert len(records) == 2 * len(components)
    write_json_atomic(output_path, {
        'schema_version': 'qwen27b-postlora-l50-confirmatory-patch-records-1.0',
        'example_id': example.example_id,
        'records': records,
    })
print('Confirmatory patch prompts complete:', len(test_examples))


## 4. Scene-level causal decision


In [ ]:
%%cell_progress 15/16 Analyze causal recovery, random specificity, and subspace content
cell_stage('Loading confirmatory patches and clean baselines')
patch_rows = []
for path in sorted(TEST_PATCH_DIR.glob('*.json')):
    if path.name == 'identity.json':
        continue
    patch_rows.extend(json.loads(path.read_text(encoding='utf-8'))['records'])
test_patch_frame = pd.DataFrame(patch_rows)
assert test_patch_frame['example_id'].nunique() == len(test_examples)

clean_rows = []
for example in test_examples:
    payload = load_clean_payload(TEST_CLEAN_DIR, example)
    for mode in ('native', 'lora'):
        record = dict(payload[f'{mode}_record'])
        record['mode'] = mode
        clean_rows.append(record)
test_clean_frame = pd.DataFrame(clean_rows)
clean_wide = test_clean_frame.pivot(
    index='example_id', columns='mode', values=['correct_margin', 'correct']
)
clean_wide.columns = ['_'.join(values) for values in clean_wide.columns]
clean_wide = clean_wide.reset_index()

prompt_effect = test_patch_frame.merge(
    clean_wide, on='example_id', validate='many_to_one'
)
prompt_effect['effect'] = np.where(
    prompt_effect['recipient_role'] == 'native_sufficiency',
    prompt_effect['correct_margin'] - prompt_effect['correct_margin_native'],
    prompt_effect['correct_margin_lora'] - prompt_effect['correct_margin'],
)
prompt_effect['accuracy_effect'] = np.where(
    prompt_effect['recipient_role'] == 'native_sufficiency',
    prompt_effect['correct'].astype(float) - prompt_effect['correct_native'].astype(float),
    prompt_effect['correct_lora'].astype(float) - prompt_effect['correct'].astype(float),
)
scene_effect = (
    prompt_effect.groupby([
        'scene_id', 'recipient_role', 'component_name',
        'component_family', 'component_rank',
    ], as_index=False).agg(
        effect=('effect', 'mean'),
        accuracy_effect=('accuracy_effect', 'mean'),
        component_norm_fraction=('component_norm_fraction', 'mean'),
        component_energy_fraction=('component_energy_fraction', 'mean'),
    )
)
scene_effect.to_csv(ANALYSIS_DIR / 'confirmatory_scene_effects.csv', index=False)

full_by_role = {
    role: scene_effect[
        (scene_effect['recipient_role'] == role)
        & (scene_effect['component_name'] == 'full_delta')
    ].set_index('scene_id')['effect']
    for role in ('native_sufficiency', 'lora_necessity')
}

summary_rows = []
for component_index, (keys, group) in enumerate(scene_effect.groupby(
    ['component_name', 'recipient_role'], sort=True
)):
    component_name, role = keys
    current = group.set_index('scene_id')['effect']
    full = full_by_role[role].reindex(current.index)
    effect_summary = bootstrap_summary(
        current, SEED + 6000 + component_index
    )
    noninferiority = bootstrap_summary(
        current - NONINFERIORITY_FRACTION * full,
        SEED + 7000 + component_index,
    )
    summary_rows.append({
        'component_name': component_name,
        'recipient_role': role,
        'component_family': group['component_family'].iloc[0],
        'component_rank': int(group['component_rank'].iloc[0]),
        **{f'effect_{key}': value for key, value in effect_summary.items()},
        'accuracy_effect': float(group['accuracy_effect'].mean()),
        'mean_norm_fraction': float(group['component_norm_fraction'].mean()),
        'mean_energy_fraction': float(group['component_energy_fraction'].mean()),
        'recovery_of_full': float(
            current.mean() / max(abs(full.mean()), 1e-8)
        ),
        'noninferiority_mean': noninferiority['mean'],
        'noninferiority_ci_low': noninferiority['ci_low'],
        'noninferiority_ci_high': noninferiority['ci_high'],
    })
confirmatory_summary = pd.DataFrame(summary_rows)
confirmatory_summary.to_csv(
    ANALYSIS_DIR / 'confirmatory_component_summary.csv', index=False
)

cell_stage('Computing empirical specificity against 32 same-rank subspaces')
empirical = {}
for role in ('native_sufficiency', 'lora_necessity'):
    selected_values = scene_effect[
        (scene_effect['recipient_role'] == role)
        & (scene_effect['component_name'] == SELECTED_NAME)
    ]['effect'].to_numpy()
    selected_mean = float(selected_values.mean())
    random_means = (
        scene_effect[
            (scene_effect['recipient_role'] == role)
            & scene_effect['component_name'].str.startswith('random_subspace_')
        ].groupby('component_name')['effect'].mean().sort_index()
    )
    assert len(random_means) == RANDOM_SUBSPACE_COUNT
    raw_p = float(
        (1 + np.sum(random_means.to_numpy() >= selected_mean))
        / (RANDOM_SUBSPACE_COUNT + 1)
    )
    selected_rows = scene_effect[
        (scene_effect['recipient_role'] == role)
        & (scene_effect['component_name'] == SELECTED_NAME)
    ]
    selected_efficiency = float(
        selected_rows['effect'].mean()
        / max(selected_rows['component_norm_fraction'].mean(), 1e-8)
    )
    random_rows = scene_effect[
        (scene_effect['recipient_role'] == role)
        & scene_effect['component_name'].str.startswith('random_subspace_')
    ]
    random_efficiency = (
        random_rows.groupby('component_name')
        .apply(
            lambda frame: frame['effect'].mean()
            / max(frame['component_norm_fraction'].mean(), 1e-8),
            include_groups=False,
        )
        .sort_index()
    )
    efficiency_p = float(
        (1 + np.sum(random_efficiency.to_numpy() >= selected_efficiency))
        / (RANDOM_SUBSPACE_COUNT + 1)
    )
    empirical[role] = {
        'selected_mean': selected_mean,
        'random_means': random_means.to_dict(),
        'raw_empirical_p': raw_p,
        'selected_effect_per_norm_fraction': selected_efficiency,
        'random_effect_per_norm_fraction': random_efficiency.to_dict(),
        'efficiency_empirical_p': efficiency_p,
    }

cell_stage('Auditing A/B/C robustness and geometric content')
permutation_summaries = {}
for role in ('native_sufficiency', 'lora_necessity'):
    permutation_summaries[role] = {}
    selected_prompts = prompt_effect[
        (prompt_effect['recipient_role'] == role)
        & (prompt_effect['component_name'] == SELECTED_NAME)
    ]
    for label_index, gold_label in enumerate(('A', 'B', 'C')):
        values = selected_prompts.loc[
            selected_prompts['gold_label'] == gold_label, 'effect'
        ]
        permutation_summaries[role][gold_label] = bootstrap_summary(
            values, SEED + 8000 + label_index
            + (0 if role == 'native_sufficiency' else 100)
        )

selected_basis_np = selected_basis.numpy()
geometry = {
    'selected_rank': SELECTED_RANK,
    'fraction_of_outcome_axis_captured': float(np.linalg.norm(
        selected_basis_np @ outcome_axis
    ) ** 2),
    'fraction_of_torque_axis_captured': float(np.linalg.norm(
        selected_basis_np @ torque_axis
    ) ** 2),
    'selected_basis_fraction_in_scene_shared_span': float(np.sum(
        (selected_basis_np @ shared_basis.T) ** 2
    ) / SELECTED_RANK),
    'selected_basis_fraction_in_mapping_residual_span': float(np.sum(
        (selected_basis_np @ mapping_basis.T) ** 2
    ) / SELECTED_RANK),
    'development_joint_delta_energy_captured': float(
        np.sum(joint_singular[:SELECTED_RANK] ** 2)
        / np.sum(joint_singular ** 2)
    ) if SELECTED_FAMILY == 'joint_svd' else None,
}

def summary_row(component_name, role):
    row = confirmatory_summary[
        (confirmatory_summary['component_name'] == component_name)
        & (confirmatory_summary['recipient_role'] == role)
    ]
    assert len(row) == 1
    return row.iloc[0]

selected_suff = summary_row(SELECTED_NAME, 'native_sufficiency')
selected_nec = summary_row(SELECTED_NAME, 'lora_necessity')
full_suff = summary_row('full_delta', 'native_sufficiency')
full_nec = summary_row('full_delta', 'lora_necessity')
permutation_robust = all(
    permutation_summaries[role][label]['ci_low'] > 0
    for role in ('native_sufficiency', 'lora_necessity')
    for label in ('A', 'B', 'C')
)
gates = {
    'full_patch_sufficiency_replicates': bool(full_suff['effect_ci_low'] > 0),
    'full_patch_necessity_replicates': bool(full_nec['effect_ci_low'] > 0),
    'selected_subspace_sufficient': bool(selected_suff['effect_ci_low'] > 0),
    'selected_subspace_necessary': bool(selected_nec['effect_ci_low'] > 0),
    'selected_sufficiency_at_least_80pct_full': bool(
        selected_suff['noninferiority_ci_low'] > 0
    ),
    'selected_necessity_at_least_80pct_full': bool(
        selected_nec['noninferiority_ci_low'] > 0
    ),
    'selected_beats_random_sufficiency': (
        empirical['native_sufficiency']['raw_empirical_p'] <= ALPHA_LEVEL
    ),
    'selected_beats_random_necessity': (
        empirical['lora_necessity']['raw_empirical_p'] <= ALPHA_LEVEL
    ),
    'selected_efficiency_beats_random_sufficiency': (
        empirical['native_sufficiency']['efficiency_empirical_p'] <= ALPHA_LEVEL
    ),
    'selected_efficiency_beats_random_necessity': (
        empirical['lora_necessity']['efficiency_empirical_p'] <= ALPHA_LEVEL
    ),
    'selected_permutation_robust': bool(permutation_robust),
}
gates['low_rank_causal_subspace_confirmed'] = all(gates.values())

decision = {
    'schema_version': 'qwen27b-postlora-l50-causal-subspace-decision-1.0',
    'selection': selection_payload,
    'confirmatory_summary': confirmatory_summary.to_dict(orient='records'),
    'random_specificity': empirical,
    'permutation_summaries': permutation_summaries,
    'subspace_geometry': geometry,
    'gates': gates,
    'interpretation_limit': (
        'Passing identifies a held-out causal state subspace under same-prompt '
        'Native/LoRA hybrid patches. It does not identify a unique circuit or '
        'show that the model explicitly multiplies weight by distance.'
    ),
}
DECISION_PATH = ANALYSIS_DIR / 'l50_causal_subspace_decision.json'
write_json_atomic(DECISION_PATH, decision)

display(confirmatory_summary[
    confirmatory_summary['component_name'].isin([
        'full_delta', SELECTED_NAME, 'selected_orthogonal_complement',
        'scene_shared_full', 'mapping_residual_full',
        'outcome_probe_rank1', 'torque_probe_rank1', 'probe_span_rank2',
    ])
].sort_values(['recipient_role', 'component_name']).round(4))
print('Selected geometry:', geometry)
print('Random empirical p:', {
    role: {
        'raw': values['raw_empirical_p'],
        'efficiency': values['efficiency_empirical_p'],
    } for role, values in empirical.items()
})
print('Frozen gates:', gates)

cell_stage('Rendering summary figure')
plt.style.use('seaborn-v0_8-whitegrid')
figure, axes = plt.subplots(2, 2, figsize=(15, 11))

rank_plot = candidate_summary[
    candidate_summary['component_family'].isin([
        'joint_svd', 'scene_shared', 'mapping_residual'
    ])
]
for family, color in (
    ('joint_svd', '#4C78A8'),
    ('scene_shared', '#54A24B'),
    ('mapping_residual', '#F58518'),
):
    current = rank_plot[rank_plot['component_family'] == family]
    axes[0, 0].plot(
        current['component_rank'], current['sufficiency_recovery'],
        marker='o', color=color, label=f'{family}: suff',
    )
    axes[0, 0].plot(
        current['component_rank'], current['necessity_recovery'],
        marker='x', linestyle='--', color=color, label=f'{family}: nec',
    )
axes[0, 0].axhline(NONINFERIORITY_FRACTION, color='black', linewidth=1)
axes[0, 0].set_xlabel('Subspace rank')
axes[0, 0].set_ylabel('Development recovery of full patch')
axes[0, 0].set_title('A. Development rank selection')
axes[0, 0].legend(fontsize=7, ncol=2)

key_components = list(dict.fromkeys([
    'full_delta', SELECTED_NAME, 'selected_orthogonal_complement',
    'scene_shared_full', 'mapping_residual_full',
    'outcome_probe_rank1', 'torque_probe_rank1', 'probe_span_rank2',
]))
effect_plot = confirmatory_summary[
    confirmatory_summary['component_name'].isin(key_components)
].pivot(index='component_name', columns='recipient_role', values='effect_mean')
effect_plot = effect_plot.reindex([name for name in key_components if name in effect_plot.index])
effect_plot.plot(kind='bar', ax=axes[0, 1], color=['#E45756', '#4C78A8'])
axes[0, 1].axhline(0, color='black', linewidth=1)
axes[0, 1].set_ylabel('Correct-margin causal effect')
axes[0, 1].set_title('B. Held-out component effects')
axes[0, 1].tick_params(axis='x', rotation=35)
axes[0, 1].legend(fontsize=8)

for role, color in (
    ('native_sufficiency', '#4C78A8'),
    ('lora_necessity', '#E45756'),
):
    random_values = np.asarray(list(empirical[role]['random_means'].values()))
    axes[1, 0].hist(random_values, bins=10, alpha=0.45, color=color, label=role)
    axes[1, 0].axvline(
        empirical[role]['selected_mean'], color=color, linewidth=2,
        linestyle='--',
    )
axes[1, 0].set_xlabel('Mean causal effect')
axes[1, 0].set_ylabel('Random subspace count')
axes[1, 0].set_title('C. Selected versus 32 matched-rank controls')
axes[1, 0].legend(fontsize=8)

geometry_labels = [
    'outcome axis', 'torque axis', 'scene-shared span', 'mapping span'
]
geometry_values = [
    geometry['fraction_of_outcome_axis_captured'],
    geometry['fraction_of_torque_axis_captured'],
    geometry['selected_basis_fraction_in_scene_shared_span'],
    geometry['selected_basis_fraction_in_mapping_residual_span'],
]
axes[1, 1].bar(geometry_labels, geometry_values, color='#72B7B2')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].set_ylabel('Fraction captured / overlap')
axes[1, 1].set_title('D. Selected-subspace geometric content')
axes[1, 1].tick_params(axis='x', rotation=25)

plt.tight_layout()
SUMMARY_FIGURE_PATH = FIGURE_DIR / 'l50_causal_subspace_summary.png'
plt.savefig(SUMMARY_FIGURE_PATH, dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
%%cell_progress 16/16 Verify the complete causal-subspace package
cell_stage('Checking required files, counts, and frozen ordering')
required_outputs = (
    PROTOCOL_LOCK_PATH, DATASET_LOCK_PATH,
    OUTPUT_ROOT / 'blind_v2_scene_manifest.jsonl',
    DEV_MANIFEST_PATH, TEST_MANIFEST_PATH,
    DEV_CLEAN_DIR / 'identity.json', DEV_CLEAN_DIR / 'index.jsonl',
    DEV_CLEAN_DIR / 'self_patch_parity.json',
    SUBSPACE_PATH, DEV_PATCH_IDENTITY_PATH,
    ANALYSIS_DIR / 'development_scene_effects.csv',
    ANALYSIS_DIR / 'development_candidate_summary.csv',
    SELECTION_PATH,
    TEST_CLEAN_DIR / 'identity.json', TEST_CLEAN_DIR / 'index.jsonl',
    RANDOM_SUBSPACE_PATH, TEST_PATCH_IDENTITY_PATH,
    ANALYSIS_DIR / 'confirmatory_scene_effects.csv',
    ANALYSIS_DIR / 'confirmatory_component_summary.csv',
    DECISION_PATH, SUMMARY_FIGURE_PATH,
)
missing = [str(path) for path in required_outputs if not Path(path).is_file()]
assert not missing, 'Missing required outputs:\n' + '\n'.join(missing)
assert len(list(DEV_CLEAN_DIR.glob('*.pt'))) == len(dev_examples)
assert len([
    path for path in DEV_PATCH_DIR.glob('*.json') if path.name != 'identity.json'
]) == len(dev_examples)
assert len(list(TEST_CLEAN_DIR.glob('*.pt'))) == len(test_examples)
assert len([
    path for path in TEST_PATCH_DIR.glob('*.json') if path.name != 'identity.json'
]) == len(test_examples)

completion = {
    'schema_version': 'qwen27b-postlora-l50-causal-subspace-complete-1.0',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'adapter_dir': str(ADAPTER_DIR),
    'fresh_blind_v2_scenes': 48,
    'development_scenes': len(dev_scene_ids),
    'confirmatory_scenes': len(test_scene_ids),
    'permutations_per_scene': 3,
    'selected_subspace': selected,
    'random_subspace_count': RANDOM_SUBSPACE_COUNT,
    'decision_path': str(DECISION_PATH),
    'summary_figure_path': str(SUMMARY_FIGURE_PATH),
    'required_outputs': [str(path) for path in required_outputs],
}
RUN_COMPLETE_PATH = OUTPUT_ROOT / 'RUN_COMPLETE.json'
write_json_atomic(RUN_COMPLETE_PATH, completion)
print('L50 causal-subspace decomposition complete:', RUN_COMPLETE_PATH)
print('Primary decision:', DECISION_PATH)
print('Summary figure:', SUMMARY_FIGURE_PATH)
